# Comparing Terraform `for_each` vs `count` for conditional resource creation

A hands-on look at how `for_each` and `count` handle conditionals in Terraform — what works, what breaks, and when to reach for each.

## Purpose

Both `for_each` and `count` let you create multiple instances of a resource from a single block. But they behave differently when you need **conditional creation** — creating some instances and skipping others based on data or logic.

This notebook walks through both approaches using the `local_file` provider (no cloud credentials needed), then compares the trade-offs.

## Setup

Make sure Terraform is installed and this notebook runs in an empty working directory.

In [ ]:
# Setup: create a clean scratch directory for this experiment
import os
import subprocess
import json
from pathlib import Path

scratch = Path("/tmp/tf-for-each-vs-count")
scratch.mkdir(parents=True, exist_ok=True)
os.chdir(str(scratch))
print(f"Working in {scratch}")

In [ ]:
# Check Terraform is available
result = subprocess.run(["terraform", "version"], capture_output=True, text=True)
if result.returncode != 0:
    print("Terraform not found — install it first")
else:
    print(result.stdout.strip())

## Using `count` for conditional creation

The classic approach: `count = condition ? 1 : 0`. If the condition is true, one instance is created; if false, zero instances.

This works well when you're creating **a single optional resource** — like a dev-only bucket or a conditional IAM policy attachment.

In [ ]:
count_main = '''
terraform {
  required_providers {
    local = {
      source = "hashicorp/local"
      version = "~> 2.0"
    }
  }
}

provider "local" {}

variable "create_config" {
  type    = bool
  default = true
}

resource "local_file" "config" {
  count    = var.create_config ? 1 : 0
  content  = "environment = production"
  filename = "${path.module}/config.ini"
}

output "file_created" {
  value = length(local_file.config) > 0 ? true : false
}
'''

with open(scratch / "count_test.tf", "w") as f:
    f.write(count_main)
print("Wrote count_test.tf")

In [ ]:
def run_terraform(cmd, expected_rc=0):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=scratch)
    if result.returncode != expected_rc:
        print(f"  RC={result.returncode}: {result.stderr.strip()[-200:]}")
    else:
        print(result.stdout.strip()[-300:] if result.stdout else "OK")
    return result

run_terraform(["terraform", "init"])

In [ ]:
# Plan with create_config = true (default)
print("=== Plan with create_config = true ===")
run_terraform(["terraform", "plan"])

In [ ]:
# Apply with create_config = true
print("=== Apply with create_config = true ===")
run_terraform(["terraform", "apply", "-auto-approve"])

import os as _os
print(f"\nFile exists: {_os.path.isfile(scratch / 'config.ini')}")

In [ ]:
# Now destroy and re-apply with create_config = false
print("=== Re-apply with create_config = false ===")
run_terraform(["terraform", "apply", "-auto-approve", "-var", "create_config=false"])

In [ ]:
print(f"File exists after disabling: {_os.path.isfile(scratch / 'config.ini')}")
# Clean up
run_terraform(["terraform", "destroy", "-auto-approv"])

**What I noticed with `count`:**
- The `count` ternary works, but accessing attributes of the resource becomes awkward — you need `local_file.config[0].filename` and have to guard against the zero-instance case.
- When `count = 0`, `local_file.config` is an empty tuple — referencing `local_file.config[0]` without a length check causes a plan-time error.
- Works well for a single optional resource where you just need a boolean on/off.

## Using `for_each` for conditional creation

With `for_each`, you pass a map or set. Conditional creation is handled by filtering the collection — instances whose keys are absent simply don't exist.

This pattern shines when you have a **set of items and need to skip some** based on a condition per item.

In [ ]:
foreach_main = '''
terraform {
  required_providers {
    local = {
      source = "hashicorp/local"
      version = "~> 2.0"
    }
  }
}

provider "local" {}

variable "files" {
  type = map(object({
    content  = string
    enabled  = bool
  }))
  default = {
    "app" = {
      content  = "app_config = true"
      enabled  = true
    }
    "db" = {
      content  = "db_host = localhost"
      enabled  = true
    }
    "cache" = {
      content  = "cache_size = 256"
      enabled  = false
    }
  }
}

resource "local_file" "configs" {
  for_each = {
    for k, v in var.files : k => v
    if v.enabled
  }
  content  = each.value.content
  filename = "${path.module}/${each.key}.ini"
}

output "enabled_files" {
  value = keys(local_file.configs)
}
'''

scratch_f = Path("/tmp/tf-foreach-test")
scratch_f.mkdir(parents=True, exist_ok=True)
os.chdir(str(scratch_f))

with open(scratch_f / "foreach_test.tf", "w") as f:
    f.write(foreach_main)
print("Wrote foreach_test.tf")

In [ ]:
def tf(cmd, expected_rc=0):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=scratch_f)
    if result.returncode != expected_rc:
        print(f"  RC={result.returncode}: {result.stderr.strip()[-200:]}")
    else:
        print(result.stdout.strip()[-400:] if result.stdout else "OK")
    return result

tf(["terraform", "init"])

In [ ]:
print("=== Plan with for_each filter ===")
tf(["terraform", "plan"])

In [ ]:
print("=== Apply ===")
tf(["terraform", "apply", "-auto-approve"])

import glob
print(f"\nCreated files: {glob.glob(str(scratch_f / '*.ini'))}")

Notice that only `app.ini` and `db.ini` were created — `cache.ini` was skipped because `enabled = false`.

In [ ]:
# Clean up
tf(["terraform", "destroy", "-auto-approve"])

**What I noticed with `for_each`:**
- The `for` expression with `if` is more expressive — I can filter on any attribute of the map values.
- Each instance has a stable key (`each.key`) so downstream references like `local_file.configs["app"].filename` are predictable and don't need index guards.
- For simple boolean on/off of a single resource, `for_each` feels like overkill — the `for` expression boilerplate outweighs the benefit.

## Head-to-head comparison

| Aspect | `count` | `for_each` |
|---|---|---|
| Conditional creation | `count = condition ? 1 : 0` | `for` expression with `if` filter |
| Single optional resource | Clean, minimal | Overkill — needs a map with one entry |
| Multiple filtered resources | Awkward — needs `count.index` mapping | Natural — each item has a key |
| Attribute access | Index-based (`res[0]`), needs length guard | Key-based (`res["key"]`), always stable |
| Refactoring risk | Changing order causes re-creation | Key stability avoids re-creation |
| Terraform state diffs | Harder to read (index-based) | Clearer (key-based) |

### When to reach for each

- **Use `count`** for a single resource that should conditionally exist (or not) as a boolean toggle.
- **Use `for_each` + for/if** when you have a collection of candidate items and need to create a subset based on per-item attributes.
- **Avoid `count`** when the order of items could change — even an insertion in the middle of a list will shift indices and trigger re-creation. `for_each`'s key-based identity avoids this entirely.

## Verify

To verify these approaches in your own infrastructure:

1. Run `terraform plan` with both approaches and a toggle — confirm the plan shows exactly the resources you expect to be created or destroyed.
2. With `count`: test the `length(resource.name) > 0 ? resource.name[0].attr : null` pattern for safe attribute access.
3. With `for_each`: test that changing a map value's attributes triggers an update in-place, not a destroy-and-recreate.
4. Use `terraform state list` after apply to see the indexed (`resource.name[0]`) vs. keyed (`resource.name["key"]`) address formats.